# Kenya Climate EDA
Exploratory Data Analysis of Kenya's climate dataset as part of the 10 Academy Week 0 challenge.

The dataset contains daily climate records from 2015 to 2026. Key columns include:
- **T2M**: Temperature at 2 meters (°C)
- **T2M_MAX / T2M_MIN**: Daily max and min temperature
- **PRECTOTCORR**: Precipitation (mm)
- **RH2M**: Relative humidity at 2 meters (%)
- **WS2M**: Wind speed at 2 meters (m/s)
- **PS**: Surface pressure
- **QV2M**: Specific humidity
- **DOY**: Day of year

## 1. Imports

In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load Data
Load the Kenya climate dataset and preview the first few rows.

In [ ]:
df = pd.read_csv('../data/kenya.csv', sep='\t')
df['Country'] = 'Kenya'
df.head()

,YEAR\tDOY\tT2M\tT2M_MAX\tT2M_MIN\tT2M_RANGE\tPRECTOTCORR\tRH2M\tWS2M\tWS2M_MAX\tPS\tQV2M,Country
0,2015\t1\t19.56\t28.99\t12.09\t16.9\t0\t45.32\t...,Kenya
1,2015\t2\t19.63\t29.77\t11.04\t18.73\t0\t38.76\...,Kenya
2,2015\t3\t20.4\t30.57\t11.71\t18.86\t0\t41.75\t...,Kenya
3,2015\t4\t21.33\t31.2\t13.02\t18.18\t3.49\t51.8...,Kenya
4,2015\t5\t20.41\t29.52\t12.38\t17.14\t1.79\t48....,Kenya


## 3. Data Overview
Check summary statistics and missing values.

In [33]:
df.describe()

,YEAR\tDOY\tT2M\tT2M_MAX\tT2M_MIN\tT2M_RANGE\tPRECTOTCORR\tRH2M\tWS2M\tWS2M_MAX\tPS\tQV2M,Country
count,4108,4108
unique,4108,1
top,2015\t1\t19.56\t28.99\t12.09\t16.9\t0\t45.32\t...,Kenya
freq,1,4108


In [34]:
df.isna().sum()

YEAR\tDOY\tT2M\tT2M_MAX\tT2M_MIN\tT2M_RANGE\tPRECTOTCORR\tRH2M\tWS2M\tWS2M_MAX\tPS\tQV2M    0
Country                                                                                     0
dtype: int64

## 4. Data Cleaning
Replace sentinel value -999 with NaN, remove duplicate rows, and forward-fill missing values.

In [35]:
df.replace(-999, np.nan, inplace=True)
df.drop_duplicates(inplace=True)
df.ffill(inplace=True)

,YEAR\tDOY\tT2M\tT2M_MAX\tT2M_MIN\tT2M_RANGE\tPRECTOTCORR\tRH2M\tWS2M\tWS2M_MAX\tPS\tQV2M,Country
0,2015\t1\t19.56\t28.99\t12.09\t16.9\t0\t45.32\t...,Kenya
1,2015\t2\t19.63\t29.77\t11.04\t18.73\t0\t38.76\...,Kenya
2,2015\t3\t20.4\t30.57\t11.71\t18.86\t0\t41.75\t...,Kenya
3,2015\t4\t21.33\t31.2\t13.02\t18.18\t3.49\t51.8...,Kenya
4,2015\t5\t20.41\t29.52\t12.38\t17.14\t1.79\t48....,Kenya
...,...,...
4103,2026\t86\t19.37\t25.4\t15.39\t10.01\t2.67\t81....,Kenya
4104,2026\t87\t19.66\t26.4\t15.24\t11.16\t0.59\t77....,Kenya
4105,2026\t88\t19.72\t26.54\t14.41\t12.13\t0.82\t77...,Kenya
4106,2026\t89\t19.68\t26.81\t13.86\t12.95\t4.59\t79...,Kenya


## 5. Feature Engineering
Create a proper date column from YEAR and DOY, then extract the Month for seasonal analysis.

In [36]:
if 'YEAR' in df.columns and 'DOY' in df.columns:
    df['date'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j')
    df['Month'] = df['date'].dt.month

## 6. Visualizations
Explore seasonal patterns in temperature, rainfall, and correlations between climate variables.

In [37]:
monthly_temp = df.groupby('Month')['T2M'].mean()
monthly_temp.plot()
plt.title('Average Monthly Temperature')
plt.show()

KeyError: 'Month'

Temperature peaks around March–April (~18°C) and drops to ~14°C in November, reflecting a highland climate pattern with cooler temperatures during the wetter months.

In [ ]:
monthly_rain = df.groupby('Month')['PRECTOTCORR'].sum()
monthly_rain.plot(kind='bar')
plt.title('Monthly Rainfall')
plt.show()

Rainfall peaks in July–August and is lowest in November–January, indicating a strong seasonal monsoon pattern with a distinct wet and dry season.

In [ ]:
sns.heatmap(df.corr(numeric_only=True), cmap='coolwarm')
plt.show()

The heatmap shows strong positive correlations between T2M, T2M_MAX, and T2M_MIN. Month correlates with rainfall (PRECTOTCORR), confirming the seasonal rainfall pattern.

## 7. Export Clean Data
Save the cleaned dataset for use in further analysis.

In [ ]:
df.to_csv('../data/kenya_clean.csv', index=False)